In [0]:
%sql
create catalog if not exists investment_pyspark;
use catalog investment_pyspark;

create schema if not exists bronze;
create volume if not exists bronze.landing;

In [0]:
from pyspark.sql.functions import current_timestamp, col
import re

In [0]:
display(
    dbutils.fs.ls("/Volumes/investment_pyspark/bronze/landing/holdings.csv/")
)

In [0]:
# 1. Source Path (Where your uploaded holdings.csv lives)
source_path = "/Volumes/investment_pyspark/bronze/landing/"

# 2. Schema Path (Databricks will create this '_schemas' folder automatically)
schema_path = "/Volumes/investment_pyspark/bronze/landing/_schemas/holdings"

# 3. Checkpoint Path (Databricks will create this '_checkpoints' folder automatically)
checkpoint_path = (
    "/Volumes/investment_pyspark/bronze/landing/_checkpoints/holdings"
)

In [0]:
df_bronze = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)



In [0]:
display(df_bronze.columns)

## Normalizeing only column names

In [0]:
def clean_column_name(column_name):
    column_name = column_name.strip()
    column_name = re.sub(r'[^a-zA-Z0-9_]', '_', column_name)
    column_name = re.sub(r'_+', '_', column_name)
    return column_name.strip('_')

In [0]:
df_bronze = df_bronze.toDF(*[clean_column_name(c) for c in df_bronze.columns])


In [0]:
df_bronze.columns

In [0]:
#df_bronze.write.format("delta").saveAsTable("investment_pyspark.bronze.holdings_raw")

In [0]:
from pyspark.sql.functions import col
df_write = df_bronze.select(
    col("Instrument").alias("Instrument"),
    col("`Qty.`").alias("Qty"),
    col("`Avg. cost`").alias("Avg_cost"),
    col("LTP").alias("LTP"),
    col("Invested").alias("Invested"),
    col("`Cur. val`").alias("Cur_val"),
    col("`P&L`").alias("P_L"),
    col("`Net chg.`").alias("Net_chg"),
    col("`Day chg.`").alias("Day_chg"),
    col("_rescued_data").alias("c9")
)
query = (df_write.writeStream.format("delta")
         .option("checkpointLocation", checkpoint_path)
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable("investment_pyspark.bronze.holdings_raw"))
query.awaitTermination()

In [0]:
df_check = spark.read.table("investment_pyspark.bronze.holdings_raw")
display(df_check)